# NB02c — Potsdam Quantitative Eval (all 6 tiles + images + HF upload)

**Runs on Kaggle T4 GPU.**

**Datasets to attach:**
- `dummyirl/sam3-weights` — SAM3 checkpoint
- `dummyirl/6isprs` — Potsdam ISPRS GeoTIFF + labels

**Secrets to attach (Settings -> Secrets):**
- `HF_TOKEN` — Hugging Face write token for `HarishDeepak/geo-prompt-peft-checkpoints`

**What this does (same as NB02b, plus):**
1. Installs env (conda + mmcv + mmseg, ~15 min)
2. Clones `HarishDeepak/rg-segearth-ov3` (our fork, aligned to teammate's protocol)
3. Runs `eval.py configs/cfg_potsdam.py` **separately per tile** (6 runs), each with
   `default_hooks.visualization.draw=True` so mmseg saves a prediction PNG per tile
4. Runs `eval.py` once more with **all 6 tiles staged together** (pooled/combined metric,
   matches teammate's `Iter(test) [6/6]` protocol) — also saves a prediction image
5. Builds **two separate comparison images per tile + combined**: one RGB-vs-Prediction,
   one GT-vs-Prediction (14 images total: 6 tiles × 2 + combined × 2)
6. Reports both: per-tile table + combined pooled table (mIoU/mAcc/mFscore + *_NoClutter)
7. Uploads metrics CSV, raw logs, and all comparison images to Hugging Face:
   `HarishDeepak/geo-prompt-peft-checkpoints`, under
   `segearth-ov3/potsdam-eval/<UTC timestamp>_nb02c-all-tiles-with-images/`

Tiles: `5_14, 5_15, 6_13, 6_14, 6_15, 7_13`


## 1 — Environment setup (~15 min, run once per session)

In [ ]:
import os

!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda_installer.sh
!bash /tmp/miniconda_installer.sh -b -p /tmp/miniconda

os.environ.pop("PYTHONPATH", None)
os.environ["PATH"] = "/tmp/miniconda/bin:" + os.environ["PATH"]

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!conda --version

In [ ]:
!/tmp/miniconda/bin/conda create -n segearth python=3.10 -y

In [ ]:
!conda run -n segearth pip install torch==2.4.0 torchvision==0.19.0 -q

In [ ]:
!conda run -n segearth pip install openmim -q
!conda run -n segearth mim install "mmcv==2.2.0" -q
!conda run -n segearth pip install "mmsegmentation==1.2.2" -q

In [ ]:
!conda run -n segearth pip install openpyxl -q

In [ ]:
%%bash
# mmseg 1.2.2 declares mmcv<2.2.0 as max; patch so version check passes
source /tmp/miniconda/bin/activate segearth
python - << 'EOF'
import pathlib
f = pathlib.Path("/tmp/miniconda/envs/segearth/lib/python3.10/site-packages/mmseg/__init__.py")
f.write_text(f.read_text().replace("MMCV_MAX = '2.2.0'", "MMCV_MAX = '2.3.0'"))
print("Patched MMCV_MAX → 2.3.0")
EOF
pip install numpy==1.26.4 -q

In [ ]:
%%bash
source /tmp/miniconda/bin/activate segearth
python - << 'EOF'
import mmcv; print("MMCV:", mmcv.__version__)
from mmcv.ops import point_sample; print("MMCV OPS OK")
from mmseg.structures import SegDataSample; print("MMSEG OK")
import torch; print("CUDA:", torch.cuda.is_available())
EOF

## 2 — Clone our fork

In [ ]:
import subprocess, os
from pathlib import Path

REPO = Path("/tmp/SegEarth-OV-3")

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
    print(f"Updated → {REPO}")
else:
    subprocess.run(
        ["git", "clone", "--depth=1",
         "https://github.com/HarishDeepak/rg-segearth-ov3", str(REPO)],
        check=True)
    print(f"Cloned → {REPO}")

os.chdir(REPO)
!conda run -n segearth pip install -r requirements.txt -q

## 3 — Stage helper

`cfg_potsdam.py` expects:
```
/kaggle/working/PotsdamEval/
  img_dir/val/         ← *_RGB.tif
  ann_dir_indexed/val/ ← *_label_noBoundary.tif (converted to 1-indexed)
```
This cell defines a `stage_tiles(tile_list)` helper that wipes and re-stages the eval
directory for a given set of tiles, then converts RGB labels → 1-indexed.

In [ ]:
import shutil
import numpy as np
from PIL import Image
from pathlib import Path

POTSDAM_INPUT = Path("/kaggle/input/datasets/dummyirl/6isprs")
EVAL_ROOT     = Path("/kaggle/working/PotsdamEval")
ALL_TILES     = ["5_14", "5_15", "6_13", "6_14", "6_15", "7_13"]

RGB_TO_IDX = {
    (255, 255, 255): 1,  # impervious surface
    (  0,   0, 255): 2,  # building
    (  0, 255, 255): 3,  # low vegetation
    (  0, 255,   0): 4,  # tree
    (255, 255,   0): 5,  # car
    (255,   0,   0): 6,  # clutter
}

def stage_tiles(tile_list):
    if EVAL_ROOT.exists():
        shutil.rmtree(EVAL_ROOT)
    img_dst   = EVAL_ROOT / "img_dir/val"
    label_dst = EVAL_ROOT / "ann_dir_indexed/val"
    img_dst.mkdir(parents=True, exist_ok=True)
    label_dst.mkdir(parents=True, exist_ok=True)

    for tile in tile_list:
        img_files   = sorted(POTSDAM_INPUT.rglob(f"*{tile}*_RGB.tif"))
        label_files = sorted(POTSDAM_INPUT.rglob(f"*{tile}*_label_noBoundary.tif"))
        for p in img_files:
            dst = img_dst / p.name
            if not dst.exists(): dst.symlink_to(p)
        for p in label_files:
            dst = label_dst / p.name
            if not dst.exists(): dst.symlink_to(p)

    # Convert RGB labels -> 1-indexed (PotsdamDataset reduce_zero_label=True: 1->0 ... 6->5)
    for lbl_path in sorted(label_dst.iterdir()):
        raw = np.array(Image.open(lbl_path))
        if raw.ndim == 2 and raw.max() <= 6 and raw.min() >= 1:
            continue
        rgb = raw[:, :, :3]
        idx = np.zeros(rgb.shape[:2], dtype=np.uint8)
        for color, cls in RGB_TO_IDX.items():
            mask = (rgb[:,:,0]==color[0]) & (rgb[:,:,1]==color[1]) & (rgb[:,:,2]==color[2])
            idx[mask] = cls
        lbl_path.unlink()
        Image.fromarray(idx).save(lbl_path)

    n_img = len(list(img_dst.iterdir()))
    n_lbl = len(list(label_dst.iterdir()))
    print(f"Staged {tile_list}: {n_img} image(s), {n_lbl} label(s)")
    return n_img, n_lbl

## 4 — Run eval PER TILE (6 separate runs, with visualization enabled)

In [ ]:
import subprocess, re
from pathlib import Path

def run_eval(tag):
    """Run eval.py for the currently staged tile(s), saving a GT-vs-prediction PNG via mmseg's visualization hook."""
    log_path  = Path(f"/kaggle/working/potsdam_eval_log_{tag}.txt")
    show_dir  = Path(f"/kaggle/working/show_dir_{tag}")
    cmd = (
        "source /tmp/miniconda/bin/activate segearth && "
        "export MPLBACKEND=Agg && "
        "cd /tmp/SegEarth-OV-3 && "
        "python eval.py configs/cfg_potsdam.py "
        "--cfg-options "
        "test_dataloader.dataset.data_root=/kaggle/working/PotsdamEval "
        "default_hooks.visualization.draw=True "
        "default_hooks.visualization.interval=1 "
        f"visualizer.save_dir={show_dir}/ "
        "visualizer.alpha=1.0 "
        f"> {log_path} 2>&1"
    )
    subprocess.run(["bash", "-c", cmd])
    return log_path.read_text(errors="replace"), show_dir

def parse_metrics(log_text):
    """Pull the final summary metrics dict mmseg prints at the end of Runner.test()."""
    metrics = {}
    for key in ["aAcc", "mIoU", "mAcc", "mFscore", "mPrecision", "mRecall",
                "mIoU_NoClutter", "mAcc_NoClutter", "mFscore_NoClutter",
                "mPrecision_NoClutter", "mRecall_NoClutter"]:
        m = re.search(rf"{key}:\s*([\d.]+)", log_text)
        if m:
            metrics[key] = float(m.group(1))
    return metrics

per_tile_results = {}
per_tile_showdirs = {}
TILES = ["5_14", "5_15", "6_13", "6_14", "6_15", "7_13"]

for tile in TILES:
    print(f"=== Evaluating tile {tile} ===")
    stage_tiles([tile])
    log_text, show_dir = run_eval(tile)
    metrics = parse_metrics(log_text)
    per_tile_results[tile] = metrics
    per_tile_showdirs[tile] = show_dir
    print(metrics)
    print()

## 5 — Run eval COMBINED (all 6 tiles staged together, matches teammate's pooled '6/6' protocol)

In [ ]:
print("=== Evaluating ALL 6 TILES COMBINED (pooled) ===")
stage_tiles(TILES)
combined_log, combined_showdir = run_eval("combined")
combined_metrics = parse_metrics(combined_log)
print(combined_metrics)

# show tail of combined log (per-class table + summary)
print("\n".join(combined_log.strip().splitlines()[-40:]))

## 6 — Summary table: per-tile + combined

In [ ]:
import pandas as pd

rows = []
for tile, m in per_tile_results.items():
    row = {"tile": tile}
    row.update(m)
    rows.append(row)
row = {"tile": "COMBINED (pooled, matches teammate protocol)"}
row.update(combined_metrics)
rows.append(row)

df = pd.DataFrame(rows)
df.to_csv("/kaggle/working/nb02c_all_tiles_summary.csv", index=False)
print("Saved: /kaggle/working/nb02c_all_tiles_summary.csv")
df

## 7 — Build GT-vs-prediction comparison images (per tile + combined)

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from pathlib import Path

PALETTE = np.array([
    [255,255,255], [0,0,255], [0,255,255],
    [0,255,0], [255,255,0], [255,0,0],
], dtype=np.uint8)
CLASS_NAMES = ["road", "building", "grass", "tree", "car", "clutter"]

IMAGES_DIR = Path("/kaggle/working/comparison_images")
IMAGES_DIR.mkdir(exist_ok=True)

def _save_two_panel(out_path, left_arr, right_arr, left_title, right_title):
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    for ax, arr, title in zip(axes, [left_arr, right_arr], [left_title, right_title]):
        ax.imshow(arr)
        ax.set_title(title, fontsize=15, pad=8)
        ax.axis("off")

    legend_patches = [mpatches.Patch(color=np.array(PALETTE[i]) / 255., label=CLASS_NAMES[i])
                       for i in range(len(CLASS_NAMES))]
    fig.legend(handles=legend_patches, loc="lower center", ncol=6,
               fontsize=11, bbox_to_anchor=(0.5, -0.04), frameon=False)

    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out_path}")

def build_comparison(tag, show_dir, img_glob_dir, gt_glob_dir):
    """Builds two separate images: RGB-vs-Prediction and GT-vs-Prediction."""
    pred_files = sorted(Path(show_dir).rglob("*.png")) if Path(show_dir).exists() else []
    img_path = next(Path(img_glob_dir).glob("*.tif"), None)
    gt_path  = next(Path(gt_glob_dir).glob("*.tif"), None)

    if not pred_files or not img_path or not gt_path:
        print(f"[{tag}] Missing prediction/image/gt — skipping. pred_files={len(pred_files)}")
        return None

    img = np.array(Image.open(img_path))[:, :, :3]
    gt  = np.array(Image.open(gt_path))
    raw = np.array(Image.open(pred_files[0]))

    w_panel  = raw.shape[1] // 2
    pred_rgb = raw[:w_panel, w_panel:, :3]
    gt_rgb   = PALETTE[gt.clip(0, 5)]

    _save_two_panel(IMAGES_DIR / f"nb02c_{tag}_rgb_vs_pred.png",
                     img, pred_rgb, "RGB image", f"Prediction ({tag})")
    _save_two_panel(IMAGES_DIR / f"nb02c_{tag}_gt_vs_pred.png",
                     gt_rgb, pred_rgb, "Ground truth", f"Prediction ({tag})")

# Per-tile images (each tile was staged alone, so img_dir/val + ann_dir_indexed/val
# only contained that single tile at the time run_eval() ran for it — restage to regenerate)
for tile in TILES:
    stage_tiles([tile])
    build_comparison(tile, per_tile_showdirs[tile],
                      "/kaggle/working/PotsdamEval/img_dir/val",
                      "/kaggle/working/PotsdamEval/ann_dir_indexed/val")

# Combined image (uses tile 6_15 as the representative RGB/GT pair, since the combined
# show_dir contains one PNG per image in the pooled val set — just show the first one found)
stage_tiles(TILES)
build_comparison("combined", combined_showdir,
                  "/kaggle/working/PotsdamEval/img_dir/val",
                  "/kaggle/working/PotsdamEval/ann_dir_indexed/val")

## 8 — Upload metrics, logs, and images to Hugging Face

Repo: `HarishDeepak/geo-prompt-peft-checkpoints`
Path: `segearth-ov3/potsdam-eval/<UTC timestamp>_nb02c-all-tiles-with-images/`

Requires `HF_TOKEN` as a Kaggle secret (Settings -> Secrets) with write access to that repo.

In [ ]:
!pip install huggingface_hub -q
from huggingface_hub import HfApi
from datetime import datetime, timezone
from pathlib import Path

HF_REPO = "HarishDeepak/geo-prompt-peft-checkpoints"
run_name = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H%M%S") + "_nb02c-all-tiles-with-images"
HF_BASE_PATH = f"segearth-ov3/potsdam-eval/{run_name}"

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    import os
    hf_token = os.environ.get("HF_TOKEN", "")

if not hf_token:
    print("HF_TOKEN not found. Add it as a Kaggle secret (Settings -> Secrets) to enable upload.")
else:
    api = HfApi()

    # metrics CSV
    api.upload_file(
        path_or_fileobj="/kaggle/working/nb02c_all_tiles_summary.csv",
        path_in_repo=f"{HF_BASE_PATH}/metrics/nb02c_all_tiles_summary.csv",
        repo_id=HF_REPO,
        repo_type="model",
        token=hf_token,
    )

    # raw per-tile + combined logs
    for log_path in sorted(Path("/kaggle/working").glob("potsdam_eval_log_*.txt")):
        api.upload_file(
            path_or_fileobj=str(log_path),
            path_in_repo=f"{HF_BASE_PATH}/logs/{log_path.name}",
            repo_id=HF_REPO,
            repo_type="model",
            token=hf_token,
        )

    # comparison images
    for img_path in sorted(Path("/kaggle/working/comparison_images").glob("*.png")):
        api.upload_file(
            path_or_fileobj=str(img_path),
            path_in_repo=f"{HF_BASE_PATH}/images/{img_path.name}",
            repo_id=HF_REPO,
            repo_type="model",
            token=hf_token,
        )

    print(f"Uploaded to: https://huggingface.co/{HF_REPO}/tree/main/{HF_BASE_PATH}")